# TP final · Opción B · Experimentos

Este notebook entrena. Lee la tabla `survey_cleaned` que deja `00_eda_cleaning.ipynb`, arma una sola matriz y entrena cuatro modelos con el mismo corte. Cada uno queda como un run en MLflow.

La etiqueta es `music_effects`: `Improve`, `No effect`, `Worsen`. La métrica que decide es el F1 macro de test, el promedio del F1 de las tres clases con el mismo peso.

Reglas de filas, ya cerradas después de la exploración:

- Se descartan las filas sin `music_effects`. La etiqueta no se imputa.
- `bpm` menor a 40 o mayor a 250 pasa a nulo. El rango no viene de la encuesta: marca tempos imposibles, como 0 o 999999999. La fila se queda.
- La mediana de las numéricas y la moda de las categóricas se calculan solo con el train, dentro del pipeline, después del corte 80/20 estratificado (`random_state=7`).
- `timestamp` no entra: es la hora en que mandó el formulario, no un hábito de escucha.

Los cuatro modelos ven la misma matriz. La comparación, el modelo elegido y las limitaciones están en `02_comparacion.ipynb`. Ese notebook no vuelve a entrenar: lee los runs de este experimento.


## Widgets

- **Puntajes de salud.** `anxiety`, `depression`, `insomnia`, `ocd`. Vacío significa que no entra ninguno. Cada uno se marca por separado. Por defecto no entra ninguno: el modelo mira hábitos de escucha, no el malestar que la persona ya declaró.


In [ ]:
%pip install -q lightgbm catboost


In [ ]:
dbutils.widgets.multiselect(
    "puntajes_salud",
    "",
    ["anxiety", "depression", "insomnia", "ocd"],
    "Puntajes de salud",
)


## Datos

La tabla ya tiene los tipos, sin `Permissions`, con los dos nombres de streaming acortados y con columnas válidas para Delta. Acá se aplican las reglas de filas.


In [ ]:
import pandas as pd

catalogo = spark.sql("SELECT current_catalog()").first()[0]
tabla = f"{catalogo}.bigdata_emanuel_rb.survey_cleaned"
pdf = spark.table(tabla).toPandas()

sin_etiqueta = pdf["music_effects"].isna() | (pdf["music_effects"].astype(str).str.strip() == "")
pdf = pdf.loc[~sin_etiqueta].copy()

bpm = pd.to_numeric(pdf["bpm"], errors="coerce")
pdf["bpm"] = bpm.where((bpm >= 40) & (bpm <= 250))

BOOLEANAS = ["while_working", "instrumentalist", "composer", "exploratory", "foreign_languages"]

def a_si_no(valor):
    if pd.isna(valor):
        return None
    if valor in (True, "Yes", "true", "True"):
        return "Yes"
    if valor in (False, "No", "false", "False"):
        return "No"
    return str(valor)

for columna in BOOLEANAS:
    pdf[columna] = pdf[columna].map(a_si_no)

print(f"tabla: {tabla}")
print(f"filas con etiqueta: {len(pdf)}")
print(pdf["music_effects"].value_counts().to_string())
print(f"bpm nulos despues de 40-250: {int(pdf['bpm'].isna().sum())}")


## Matriz y corte

Hábitos de la tabla limpia. Los puntajes marcados en el widget se suman como columnas. El corte es 80/20 estratificado, `random_state=7`, igual para los cuatro.


In [ ]:
from sklearn.model_selection import train_test_split

PUNTAJES = ["anxiety", "depression", "insomnia", "ocd"]
NUMERICAS = ["age", "hours_per_day", "bpm"]
elegidos = [nombre.strip() for nombre in dbutils.widgets.get("puntajes_salud").split(",") if nombre.strip()]
desconocidos = [nombre for nombre in elegidos if nombre not in PUNTAJES]
if desconocidos:
    raise ValueError(f"Puntaje desconocido: {desconocidos}")

columnas = [nombre for nombre in pdf.columns if nombre not in ("timestamp", "music_effects", *PUNTAJES)]
columnas = columnas + [nombre for nombre in PUNTAJES if nombre in elegidos]

X = pdf[columnas]
y = pdf["music_effects"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7
)
print(f"columnas: {len(columnas)}")
print(f"puntajes: {elegidos or 'ninguno'}")
print(f"train: {len(X_train)}  test: {len(X_test)}")
print(y_test.value_counts().to_string())


## Modelos

Un pipeline por modelo: imputar, escalar las numéricas, one-hot de las categóricas, y el estimador. La imputación queda dentro del pipeline para que la mediana y la moda salgan del train.

| Run | Modelo | Balanceo |
|---|---|---|
| `logistica` | Regresión logística, `C=1` | no |
| `random_forest` | Random forest, 100 árboles | `class_weight='balanced'` |
| `lightgbm` | LightGBM, 100 árboles | `class_weight='balanced'` |
| `catboost` | CatBoost, 100 iteraciones | `auto_class_weights='Balanced'` |

Cada run va al experimento `/Users/<usuario>/bigdata_emanuel_rb_tp_final`. Se guardan parámetros, el F1 macro de train y de test, el F1 de cada clase, la accuracy de test, la matriz de confusión y el pipeline.


In [ ]:
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

mlflow.autolog(disable=True)
usuario = spark.sql("SELECT current_user()").first()[0]
experimento = f"/Users/{usuario}/bigdata_emanuel_rb_tp_final"
mlflow.set_experiment(experimento)

CLASES = ["Improve", "No effect", "Worsen"]

def preprocesar(columnas):
    numericas = [nombre for nombre in columnas if nombre in NUMERICAS or nombre in PUNTAJES]
    categoricas = [nombre for nombre in columnas if nombre not in numericas]
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numericas),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categoricas),
    ])

def figura_matriz(y_real, y_pred, titulo):
    matriz = confusion_matrix(y_real, y_pred, labels=CLASES)
    fig, ax = plt.subplots(figsize=(4.4, 3.6))
    ax.imshow(matriz, cmap="Blues")
    for fila in range(matriz.shape[0]):
        for col in range(matriz.shape[1]):
            valor = matriz[fila, col]
            ax.text(col, fila, str(valor), ha="center", va="center",
                    color="white" if valor > matriz.max() / 2 else "black")
    ax.set_xticks(range(3), CLASES, rotation=15)
    ax.set_yticks(range(3), CLASES)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Realidad")
    ax.set_title(titulo, fontsize=10)
    fig.tight_layout()
    return fig

modelos = {
    "logistica": LogisticRegression(C=1, max_iter=1000),
    "random_forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=7, n_jobs=-1
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100, class_weight="balanced", random_state=7, verbosity=-1
    ),
    "catboost": CatBoostClassifier(
        iterations=100, auto_class_weights="Balanced", random_state=7,
        verbose=0, allow_writing_files=False,
    ),
}

for nombre, modelo in modelos.items():
    pipeline = Pipeline([
        ("preprocesamiento", preprocesar(list(X_train.columns))),
        ("modelo", modelo),
    ])
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({
            "modelo": nombre,
            "puntajes_salud": ",".join(elegidos) if elegidos else "ninguno",
            "n_columnas": len(X_train.columns),
            "train_filas": len(X_train),
            "test_filas": len(X_test),
            "corte": "80/20 estratificado",
            "random_state": 7,
            "tabla": tabla,
        })
        pipeline.fit(X_train, y_train)
        pred_test = pipeline.predict(X_test)
        pred_train = pipeline.predict(X_train)
        f1_clases = f1_score(y_test, pred_test, average=None, labels=CLASES, zero_division=0)
        metricas = {
            "test_f1_macro": float(f1_score(y_test, pred_test, average="macro", zero_division=0)),
            "train_f1_macro": float(f1_score(y_train, pred_train, average="macro", zero_division=0)),
            "test_accuracy": float(accuracy_score(y_test, pred_test)),
            "test_f1_improve": float(f1_clases[0]),
            "test_f1_no_effect": float(f1_clases[1]),
            "test_f1_worsen": float(f1_clases[2]),
        }
        mlflow.log_metrics(metricas)
        fig = figura_matriz(y_test, pred_test, nombre)
        mlflow.log_figure(fig, "matriz_confusion.png")
        plt.close(fig)
        firma = infer_signature(X_test, pipeline.predict(X_test.head(5)))
        kwargs = dict(
            sk_model=pipeline,
            signature=firma,
            input_example=X_test.head(5),
            serialization_format="cloudpickle",
        )
        if int(mlflow.__version__.split(".")[0]) >= 3:
            info = mlflow.sklearn.log_model(name="model", **kwargs)
        else:
            info = mlflow.sklearn.log_model(artifact_path="model", **kwargs)
        mlflow.set_tag("model_uri", info.model_uri)
    print(nombre, {clave: round(valor, 4) for clave, valor in metricas.items()})

print("experimento:", experimento)
